### Phase 1: Environment Setup and Data Loading
In this section, we import the pandas library to handle our dataframes. We are loading the three pillars of our research:

1. Apple Financials (Corporate Performance)

2. US GDP Growth (Macroeconomic Health)

3. Oil Prices (Energy Market Influence)

In [1]:
import pandas as pd
import os

# Load the raw data files
# Ensure these names match the ones in your raw_data folder!
apple = pd.read_csv('raw_data/apple_financials.csv')
gdp = pd.read_csv('raw_data/us_gdp_growth.csv')
oil = pd.read_csv('raw_data/oil_prices.csv')

print("Files loaded successfully!")

Files loaded successfully!


The following script performs Data Wrangling and Normalization to prepare disparate time-series data for analysis in EViews.

1. Data Sources
Corporate: Apple Annual Financials (Source: SEC Filings/Kaggle).

Growth: US Real GDP Growth (Source: FRED, ID: A191RL1A225NBEA).

Energy: Global Crude Oil Prices (Source: FRED, ID: POILWTIUSDM).

2. Technical Challenges Addressed
Date Alignment: GDP data begins in 1930, while Apple financials begin in the 1990s. We use an Inner Join to align these timelines.

Variable Normalization: We converted complex FRED database codes into human-readable econometric variables.

Data Integrity: Implemented a "Read-Only" raw data architecture to ensure reproducibility.

In [3]:
print("GDP Columns:", gdp.columns.tolist())
print("Oil Columns:", oil.columns.tolist())
print("Apple Columns:", apple.columns.tolist())

GDP Columns: ['observation_date', 'A191RL1A225NBEA']
Oil Columns: ['observation_date', 'POILWTIUSDM']
Apple Columns: ['Unnamed: 0', 'Net sales', 'Net income', 'Earnings per  common and  common  equivalent  share', 'Earnings per diluted share', 'Cash dividends  declared  per common  share', 'Common and  common  equivalent  shares used  in the  calculations  of basic earnings per  share (Per Thousand)', 'Common and  common  equivalent  shares used  in the  calculations  of diluted earnings per  share (Per Thousand)', 'Cash, cash  equivalents,  and short-term  investments', 'Total assets', 'Commercial paper', 'Long-term debt', 'Other long-term obligations', 'Other non-current liabilities', 'Deferred tax \n liabilities', 'Total Liabilities', "Shareholder's Equity"]


In [4]:
import pandas as pd
import os

# --- Step 1: Loading Raw Data ---
# We maintain a 'raw_data' folder to preserve the original source files.
apple = pd.read_csv('raw_data/apple_financials.csv')
gdp = pd.read_csv('raw_data/us_gdp_growth.csv')
oil = pd.read_csv('raw_data/oil_prices.csv')

# --- Step 2: Time-Series Extraction ---
# Extracting the Year from 'observation_date' to create a common join key.
gdp['Year'] = pd.to_datetime(gdp['observation_date']).dt.year
oil['Year'] = pd.to_datetime(oil['observation_date']).dt.year
apple = apple.rename(columns={'Unnamed: 0': 'Year'})

# --- Step 3: Feature Selection & Renaming ---
# Selecting 'Net Income' as the dependent variable and renaming FRED codes.
gdp_final = gdp[['Year', 'A191RL1A225NBEA']].rename(columns={'A191RL1A225NBEA': 'GDP_Growth'})
oil_final = oil[['Year', 'POILWTIUSDM']].rename(columns={'POILWTIUSDM': 'Oil_Price'})

# --- Step 4: Master Merge ---
# Creating the final dataset for Econometric Testing (Stationarity & Cointegration).
master_df = apple[['Year', 'Net income']].merge(gdp_final, on='Year').merge(oil_final, on='Year')

# --- Step 5: Exporting for EViews ---
if not os.path.exists('processed_data'):
    os.makedirs('processed_data')
    
master_df.to_csv('processed_data/Apple_Macro_Master.csv', index=False)

print("✅ Master Dataset Created Successfully!")
master_df.head()

✅ Master Dataset Created Successfully!


,Year,Net income,GDP_Growth,Oil_Price
0,1992,$ 530.37,3.5,18.806957
1,1992,$ 530.37,3.5,19.071500
2,1992,$ 530.37,3.5,18.931818
3,1992,$ 530.37,3.5,20.244545
4,1992,$ 530.37,3.5,20.999048


### The "Final Polish" Script
Run this in a new cell. It will collapse the monthly data into a clean, annual average so EViews can run a proper time-series model.

In [7]:
import numpy as np

# 1. Strip whitespace first
master_df['Net income'] = master_df['Net income'].str.strip()

# 2. Handle the Parentheses (Negative Numbers)
# If a value starts with '(', we remove '(' and ')', add '-', and clean the rest
def clean_currency(value):
    if pd.isna(value) or value == '':
        return np.nan
    
    clean_val = str(value).replace('$', '').replace(',', '').strip()
    
    if clean_val.startswith('(') and clean_val.endswith(')'):
        clean_val = '-' + clean_val.replace('(', '').replace(')', '')
        
    return float(clean_val)

# 3. Apply the function to the column
master_df['Net income'] = master_df['Net income'].apply(clean_currency)

# 4. Now run the GroupBy to get Annual data
annual_master = master_df.groupby('Year').mean(numeric_only=True).reset_index()

# 5. Save the final version
annual_master.to_csv('processed_data/Apple_Macro_Annual.csv', index=False)

print("✅ Success! Handled financial losses (negative values).")
print(annual_master.head())

✅ Success! Handled financial losses (negative values).
   Year  Net income  GDP_Growth  Oil_Price
0  1992      530.37         3.5  20.582609
1  1993       86.59         2.7  18.474046
2  1994      310.18         4.0  17.167957
3  1995      424.00         2.7  18.414001
4  1996     -816.00         3.8  22.151895
